In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST
from torchvision.utils import make_grid
from skimage.metrics import structural_similarity
from torchmetrics.image.fid import FrechetInceptionDistance

TRAIN = True
SEED, BATCH, FID_N = 42, 128, 2048
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CKPT_DIR, METRIC_DIR = ROOT / "results/checkpoints/custom_ae_vae", ROOT / "results/metrics/custom_ae_vae"
CKPT_DIR.mkdir(parents=True, exist_ok=True); METRIC_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)

train_raw = FashionMNIST(ROOT / "data", train=True, download=True)
test_raw = FashionMNIST(ROOT / "data", train=False, download=True)
images = torch.cat([train_raw.data, test_raw.data]).unsqueeze(1).float() / 255.0
labels = torch.cat([train_raw.targets, test_raw.targets])

with np.load(ROOT / "results/metrics/fashion_mnist_seed42.npz") as f:
    splits = {k: TensorDataset(images[f[k]], labels[f[k]]) for k in ("train", "val", "test")}

def get_loader(name):
    return DataLoader(splits[name], BATCH, shuffle=(name == "train"), generator=torch.Generator().manual_seed(SEED))

In [2]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.decoder(self.encoder(x))

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mean, self.logvar = nn.Linear(256, 64), nn.Linear(256, 64)
        self.decoder = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Unflatten(1, (1, 28, 28)))
    def encode(self, x):
        h = self.encoder(x)
        return self.mean(h), self.logvar(h)
    def forward(self, x):
        return self.decoder(self.encode(x)[0]).sigmoid()

models = {"Vanilla AE": Autoencoder().to(DEVICE), "Sparse AE": Autoencoder().to(DEVICE), "VAE": VAE().to(DEVICE)}
slugs = {"Vanilla AE": "vanilla_ae", "Sparse AE": "sparse_ae", "VAE": "vae"}
chosen = {}

if TRAIN:
    for name, model in models.items():
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        best, hist = float("inf"), []
        for epoch in range(1, 31):
            model.train()
            tr_sum, tr_n = 0.0, 0
            for x, _ in get_loader("train"):
                x = x.to(DEVICE)
                if name == "VAE":
                    mu, logvar = model.encode(x)
                    z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)
                    loss = F.binary_cross_entropy_with_logits(model.decoder(z), x, reduction="sum") / len(x) + (mu.square() + logvar.exp() - 1 - logvar).sum() / (2 * len(x))
                else:
                    z = model.encoder(x)
                    loss = F.mse_loss(model.decoder(z), x) + (0.001 * z.abs().mean() if name == "Sparse AE" else 0.0)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                tr_sum += loss.item() * len(x); tr_n += len(x)
            model.eval()
            with torch.no_grad():
                val_mse = sum(F.mse_loss(model(vx.to(DEVICE)), vx.to(DEVICE), reduction="sum").item() for vx, _ in get_loader("val")) / (len(splits["val"]) * 784)
            hist.append({"epoch": epoch, "train_loss": tr_sum / tr_n, "val_mse": val_mse})
            if val_mse < best:
                best, chosen[name] = val_mse, epoch
                best_w = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                torch.save({"epoch": epoch, "val_mse": val_mse, "model_state_dict": best_w}, CKPT_DIR / f"{slugs[name]}_best.pt")
            if epoch % 5 == 0 or epoch == 1:
                print(f"[{name}] Epoch {epoch:02d}/30 | Train: {tr_sum/tr_n:.6f} | Val MSE (piksel^2): {val_mse:.6f} | Best: {chosen[name]}")
        model.load_state_dict(best_w)
        pd.DataFrame(hist).to_csv(METRIC_DIR / f"{slugs[name]}_history.csv", index=False)
    (METRIC_DIR / "selected_epochs.json").write_text(json.dumps(chosen, indent=2))
else:
    for name in models:
        ckpt = torch.load(CKPT_DIR / f"{slugs[name]}_best.pt", map_location="cpu", weights_only=True)
        models[name].load_state_dict(ckpt["model_state_dict"]); models[name].eval(); chosen[name] = ckpt["epoch"]

display(pd.Series(chosen, name="Epoch terpilih").to_frame())

[Vanilla AE] Epoch 01/30 | Train: 0.034843 | Val MSE (piksel^2): 0.019458 | Best: 1
[Vanilla AE] Epoch 05/30 | Train: 0.012320 | Val MSE (piksel^2): 0.012004 | Best: 5
[Vanilla AE] Epoch 10/30 | Train: 0.009924 | Val MSE (piksel^2): 0.009936 | Best: 10
[Vanilla AE] Epoch 15/30 | Train: 0.008820 | Val MSE (piksel^2): 0.008932 | Best: 15
[Vanilla AE] Epoch 20/30 | Train: 0.008254 | Val MSE (piksel^2): 0.008433 | Best: 20
[Vanilla AE] Epoch 25/30 | Train: 0.007915 | Val MSE (piksel^2): 0.008117 | Best: 25
[Vanilla AE] Epoch 30/30 | Train: 0.007690 | Val MSE (piksel^2): 0.007894 | Best: 30
[Sparse AE] Epoch 01/30 | Train: 0.035499 | Val MSE (piksel^2): 0.019832 | Best: 1
[Sparse AE] Epoch 05/30 | Train: 0.013080 | Val MSE (piksel^2): 0.012128 | Best: 5
[Sparse AE] Epoch 10/30 | Train: 0.010587 | Val MSE (piksel^2): 0.009984 | Best: 10
[Sparse AE] Epoch 15/30 | Train: 0.009538 | Val MSE (piksel^2): 0.009100 | Best: 15
[Sparse AE] Epoch 20/30 | Train: 0.008993 | Val MSE (piksel^2): 0.008651 

,Epoch terpilih
Vanilla AE,30
Sparse AE,30
VAE,29


In [ ]:
@torch.inference_mode()
def calc_fid_tm(real_x, fake_x):
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    for b in real_x.split(128): fid.update(b.to(DEVICE).repeat(1, 3, 1, 1), real=True)
    for b in fake_x.split(128): fid.update(b.to(DEVICE).repeat(1, 3, 1, 1), real=False)
    return float(fid.compute())

def calc_latency(fn, runs=30):
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    for _ in range(5): fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): fn()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / runs

test_x, test_y = splits["test"].tensors
recons, rows, interps = {}, {}, []
alphas = torch.linspace(0.0, 1.0, 6, device=DEVICE).unsqueeze(1)
i1, i2 = int((test_y == 0).nonzero(as_tuple=True)[0][0]), int((test_y == 9).nonzero(as_tuple=True)[0][0])

with torch.inference_mode():
    for name, model in models.items():
        pred = torch.cat([model(b.to(DEVICE)).cpu() for b in test_x.split(BATCH)])
        recons[name] = pred
        mse = (test_x - pred).square().mean(dim=(1, 2, 3))
        psnr = (-10.0 * mse.clamp_min(1e-12).log10()).mean().item()
        ssim = float(np.mean([structural_similarity(a[0], b[0], data_range=1.0, win_size=7) for a, b in zip(test_x.numpy(), pred.numpy())]))
        rows[name] = {"Epoch": chosen[name], "MSE (piksel^2)": mse.mean().item(), "PSNR (dB)": psnr, "SSIM": ssim}
        z1 = model.encode(test_x[i1:i1+1].to(DEVICE))[0] if name == "VAE" else model.encoder(test_x[i1:i1+1].to(DEVICE))
        z2 = model.encode(test_x[i2:i2+1].to(DEVICE))[0] if name == "VAE" else model.encoder(test_x[i2:i2+1].to(DEVICE))
        dec = model.decoder((1 - alphas) * z1 + alphas * z2)
        interps.append((dec.sigmoid() if name == "VAE" else dec).cpu())

    rng = torch.Generator(device=DEVICE).manual_seed(SEED)
    vae_gen = models["VAE"].decoder(torch.randn((FID_N, 64), device=DEVICE, generator=rng)).sigmoid().cpu()
    vae_fid = calc_fid_tm(test_x[:FID_N], vae_gen)
    vae_lat = calc_latency(lambda: models["VAE"].decoder(torch.randn((64, 64), device=DEVICE)).sigmoid())

eval_df = pd.DataFrame(rows).T.round({"Epoch": 0, "MSE (piksel^2)": 6, "PSNR (dB)": 4, "SSIM": 4})
eval_df.to_csv(METRIC_DIR / "test_reconstruction_metrics.csv")
pd.DataFrame([{"Model": "VAE", "FID Test (N=2048)": vae_fid, "Latensi Batch 64 (ms)": vae_lat}]).to_csv(METRIC_DIR / "vae_generative_metrics.csv", index=False)

display(eval_df)
print(f"VAE Test FID (TorchMetrics, N={FID_N}): {vae_fid:.4f} | Latensi Batch 64: {vae_lat:.3f} ms")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, name in zip(axes, models):
    hdf = pd.read_csv(METRIC_DIR / f"{slugs[name]}_history.csv")
    ax.plot(hdf["epoch"], hdf["train_loss"], label="Train Loss")
    ax.plot(hdf["epoch"], hdf["val_mse"], label="Val MSE (piksel^2)")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

idx8 = [int((test_y == c).nonzero(as_tuple=True)[0][0]) for c in range(8)]
plt.figure(figsize=(8, 6)); plt.imshow(make_grid(torch.cat([test_x[idx8]] + [recons[m][idx8] for m in models]), nrow=8, padding=2).permute(1, 2, 0).numpy(), cmap="gray"); plt.axis("off"); plt.show()
plt.figure(figsize=(8, 4.5)); plt.imshow(make_grid(torch.cat(interps), nrow=6, padding=2).permute(1, 2, 0).numpy(), cmap="gray"); plt.axis("off"); plt.show()

,Epoch,MSE (piksel^2),PSNR (dB),SSIM
Vanilla AE,30.0,0.007779,22.1497,0.7981
Sparse AE,30.0,0.008076,21.9816,0.7931
VAE,29.0,0.013111,19.6557,0.7061


VAE Test FID (TorchMetrics, N=2048): 95.0123 | Latensi Batch 64: 0.250 ms
